#### env (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< sm'_operators
:< unsupported

## rust

In [ ]:
:! var_error =
    `(
        rust::rust_type
            `($'std_env_VarError')
            `($'std::env::VarError')
            "type std_env_VarError = class end"
    )

:= get_environment_variable_comptime (var : string) : string =
    run_target_args (\ () => var) \|
        | Rust => \ var =>
            :< rust_operators
            !\($'"option_env\!(\\\"" + !var + "\\\").unwrap_or(\\\"\\\")"')
            |> sm'::ref_to_std_string
            |> sm'::from_std_string
        | target => \ _ => null ()

## python

In [ ]:
:! os_environ = any

:= os_environ () : os_environ =
    backend_switch {
        Gleam = \ () => "env.os_environ" |> failwith : os_environ
        Lua = \ () => "env.os_environ" |> failwith : os_environ
        Python = \ () =>
            global "import os"
            $'os.environ' : os_environ
    }

:= environ_get (key : string) (os_environ : os_environ) : string =
    backend_switch {
        Gleam = \ () => "env.environ_get" |> failwith : string
        Lua = \ () => "env.environ_get" |> failwith : string
        Python = \ () =>
            $'!os_environ.get(!key)' : string
    }

## env

In [ ]:
:~ get_environment_variable (var : string) : string =
    run_target_args (\ () => var) \|
        | Zig => \ _ =>
            $'(std.process.Environ.getAlloc(spiral_environ, spiral_gpa, !var) catch "")' : string
        | Gleam _ => \ var =>
            backend_switch {
                Gleam = \ () =>
                    global "import envoy"
                    global "import gleam/result"
            }
            $'!var |> envoy.get |> result.unwrap("")' : string
        | Fsharp => \ var =>
            var
            |> $'System.Environment.GetEnvironmentVariable'
            |> optionm'::of_obj
            |> optionm'::unbox
            |> optionm'::default_value ""
        | Lua => \ var =>
            $'os.getenv(!var) or ""' : string
        | Pascal => \ var =>
            $'GetEnvironmentVariable(!var)' : string
        | Rust => \ var =>
            $'Rc::<str>::from(std::env::var(&*!var).unwrap_or_default())' : string
        | TypeScript => \ var =>
            $'(process.env[!var] ?? "")' : string
        | Cuda _ => \ var =>
            os_environ ()
            |> environ_get var
            |> optionm'::of_obj
            |> optionm'::unbox
            |> optionm'::default_value ""
        | C => \ var =>
            global "#include <stdlib.h>"
            global "#include <string.h>"
            $'({ const char * spiral_env = getenv((!var)->ptr); \
                \spiral_env ? StringLit((uint32_t)strlen(spiral_env) + 1, (char *)spiral_env) : \
                \StringLit(1, ""); })' : string
        | target => \ var =>
            ~~ "env.get_environment_variable / target: " ++#? target ++# " / var: " ++#? var |> failwith

#### get_environment_variable_reads_set_and_unset_variables (test)

In [1]:
///- --test --symbol
///> c
///> rust

:= ~unset = "SPIRAL_SURELY_UNSET_VARIABLE_1102"
(get_environment_variable unset, sm'::length (get_environment_variable "PATH") > 0i32) |> _assert_eq ("", !+)

.c (c)
__assert_eq { expected = (, true) }
.rs (rust)
__assert_eq { expected = (, true) }

0

In [ ]:
:~ set_environment_variable (var : string) (value : string) : () =
    run_target_args (\ () => var, value) \|
        | Gleam _ => \ var, value =>
            backend_switch {
                Gleam = \ () =>
                    global "import envoy"
            }
            $'envoy.set(!var, !value)' : ()
        | Fsharp => \ var, value =>
            $'System.Environment.SetEnvironmentVariable(!var, !value)' : ()
        | Rust => \ var, value =>
            $'unsafe { std::env::set_var(&*!var, &*!value) }' : ()
        | TypeScript => \ var, value =>
            $'process.env[!var] = !value' : ()
        | target => \ var, _ =>
            ~~ "env.set_environment_variable / target: " ++#? target ++# " / var: " ++#? var |> failwith

:~ remove_environment_variable (var : string) : () =
    run_target_args (\ () => var) \|
        | Gleam _ => \ var =>
            backend_switch {
                Gleam = \ () =>
                    global "import envoy"
            }
            $'envoy.unset(!var)' : ()
        | Fsharp => \ var =>
            $'System.Environment.SetEnvironmentVariable(!var, null)' : ()
        | Rust => \ var =>
            $'unsafe { std::env::remove_var(&*!var) }' : ()
        | TypeScript => \ var =>
            $'delete process.env[!var]' : ()
        | target => \ var =>
            ~~ "env.remove_environment_variable / target: " ++#? target ++# " / var: " ++#? var |> failwith

#### set_environment_variable (test)

In [2]:
///- --test --symbol
///> gleam
///> rust
///> typescript

set_environment_variable "SPIRAL_ENV_SET_PROBE" "set"
"SPIRAL_ENV_SET_PROBE"
|> get_environment_variable
|> _assert_eq "set"

.gleam (gleam)
__assert_eq { expected = set }
.rs (rust)
__assert_eq { expected = set }
.ts (typescript)
__assert_eq { expected = set }

0

In [ ]:
:= removed_probe () : string =
    set_environment_variable "SPIRAL_ENV_REMOVE_PROBE" "set"
    remove_environment_variable "SPIRAL_ENV_REMOVE_PROBE"
    get_environment_variable "SPIRAL_ENV_REMOVE_PROBE"

#### remove_environment_variable (test)

In [3]:
///- --test --symbol
///> gleam
///> rust
///> typescript

removed_probe () |> _assert_eq ""

.gleam (gleam)
__assert_eq { expected =  }
.rs (rust)
__assert_eq { expected =  }
.ts (typescript)
__assert_eq { expected =  }

0

#### get_environment_variable (test)

In [4]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

"PATH2"
|> get_environment_variable
|> sm'::length
|> _assert_eq 0i32

.gleam (gleam)
__assert_eq { expected = 0 }
.lua (lua)
__assert_eq { expected = 0 }
.py (cuda)
__assert_eq { expected = 0 }
.rs (rust)
__assert_eq { expected = 0 }
.ts (typescript)
__assert_eq { expected = 0 }

0

In [5]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

"PATH"
|> get_environment_variable
|> sm'::length
|> \ x =>
    ? x > 0i32
    ?> 1
    |: 0
    |> _assert_eq 1i32

.gleam (gleam)
__assert_eq { expected = 1 }
.lua (lua)
__assert_eq { expected = 1 }
.py (cuda)
__assert_eq { expected = 1 }
.rs (rust)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }

0

In [ ]:
:= try_get_environment_variable (var : string) : option string =
    := value = var |> get_environment_variable
    ? value = "" ?> None |: Some value

:= variable_is_set (var : string) : i32 =
    ?= var |> try_get_environment_variable ?|
    | Some _ => 1i32
    | None => 0i32

#### try_get_environment_variable_is_none_when_unset (test)

In [6]:
///- --test --symbol
///> gleam
///> lua
///> rust
///> typescript

(variable_is_set "PATH2", variable_is_set "PATH") |> _assert_eq (0i32, 1i32)

.gleam (gleam)
__assert_eq { expected = (0, 1) }
.lua (lua)
__assert_eq { expected = (0, 1) }
.rs (rust)
__assert_eq { expected = (0, 1) }
.ts (typescript)
__assert_eq { expected = (0, 1) }

0

In [ ]:
:= env_gleam_argv () =
    global "import gleam/erlang/charlist"
    global "import gleam/list"
    global "import gleam/result"
    global "@external(erlang, \"init\", \"get_plain_arguments\")"
    global "fn spiral_env_plain_arguments() -> List(charlist.Charlist)"

:= argument_offset () : i32 =
    run_target \|
        | TypeScript => \ () => 2i32
        | Gleam => \ () => 0i32
        | _ => \ () => 1i32

:= host_argument_count () : i32 =
    run_target \|
        | Rust => \ () => ($'std::env::args().count()' : convert::unativeint) |> convert::convert
        | Fsharp => \ () => $'System.Environment.GetCommandLineArgs().Length' : i32
        | TypeScript => \ () => $'process.argv.length' : i32
        | Cuda _ => \ () =>
            global "import sys"
            $'len(sys.argv)' : i32
        | Gleam => \ () =>
            env_gleam_argv ()
            $'list.length(spiral_env_plain_arguments())' : i32
        | target => \ () => "env.host_argument_count / target: " ++#? target |> failwith

:= host_argument (position : i32) : string =
    run_target_args (\ () => position) \|
        | Rust => \ position =>
            := position : convert::unativeint = convert::convert position
            $'Rc::<str>::from(std::env::args().nth(!position).unwrap_or_default())' : string
        | Fsharp => \ position => $'System.Environment.GetCommandLineArgs().[!position]' : string
        | TypeScript => \ position => $'process.argv[!position]' : string
        | Cuda _ => \ position =>
            global "import sys"
            $'sys.argv[!position]' : string
        | Gleam => \ position =>
            env_gleam_argv ()
            := rest = $'list.drop(spiral_env_plain_arguments(), !position)' : $'List(charlist.Charlist)'
            $'result.unwrap(result.map(list.first(!rest), charlist.to_string), "")' : string
        | target => \ _ => "env.host_argument / target: " ++#? target |> failwith

:= exit_with (code : i32) : () =
    run_target_args (\ () => code) \|
        | Rust => \ code => $'std::process::exit(!code)' : ()
        | Fsharp => \ code => $'exit !code' : ()
        | TypeScript => \ code => $'process.exit(!code)' : ()
        | Cuda _ => \ code =>
            global "import sys"
            $'sys.exit(!code)' : ()
        | target => \ _ => "env.exit_with / target: " ++#? target |> failwith

:~ command_line_argument_count () : i32 =
    host_argument_count () - argument_offset ()

:~ command_line_argument (index : i32) : string =
    index + argument_offset () |> host_argument

:= try_command_line_argument (index : i32) : option string =
    ? index >= 0 && index < command_line_argument_count ()
    ?> index |> command_line_argument |> Some
    |: None

:= argument_is_set (index : i32) : i32 =
    ?= index |> try_command_line_argument ?|
    | Some _ => 1i32
    | None => 0i32

#### command_line_arguments_outside_the_range_are_none (test)

In [7]:
///- --test --symbol
///> cuda
///> rust
///> typescript

:= count = command_line_argument_count ()
(count >= 0i32, argument_is_set (-1i32), argument_is_set count)
|> _assert_eq (!+, 0i32, 0i32)

.py (cuda)
__assert_eq { expected = (true, 0, 0) }
.rs (rust)
__assert_eq { expected = (true, 0, 0) }
.ts (typescript)
__assert_eq { expected = (true, 0, 0) }

0

### command_line_arguments

In [ ]:
:~ ^ arguments_from (index : i32) : list string =
    ?= try_command_line_argument index ?|
    | None => []
    | Some argument => argument :: arguments_from (index + 1i32)

:= command_line_arguments () : list string = arguments_from 0i32

:= head_is_first_argument (head : string) : bool =
    ?= try_command_line_argument 0i32 ?|
    | Some a => a = head
    | None => !-

:= first_argument_matches (listed : list string) : bool =
    ?= listed ?|
    | head :: _ => head_is_first_argument head
    | [] => command_line_argument_count () = 0i32

#### command_line_arguments_list_every_argument_in_order (test)

In [8]:
///- --test --symbol
///> rust
///> gleam

:= listed = command_line_arguments ()
(listm::length listed, first_argument_matches listed) |> _assert_eq (command_line_argument_count (), !+)

.rs (rust)
__assert_eq { expected = (0, true) }
.gleam (gleam)
__assert_eq { expected = (0, true) }

0

### command_line_arguments_from

In [ ]:
:~ ^ command_line_arguments_back (first : i32) (at : i32) (arguments : list string) : list string =
    ? at < first ?> arguments
    |: command_line_arguments_back first (at - 1i32) (command_line_argument at :: arguments)

:~ command_line_arguments_from (index : i32) : list string =
    command_line_arguments_back (? index < 0i32 ?> 0i32 |: index) (command_line_argument_count () - 1i32) []

:= argument_count_from (index : i32) : i32 = listm::length (command_line_arguments_from index)

#### command_line_arguments_from_cover_the_rest_of_the_line (test)

In [9]:
///- --test --symbol
///> rust

:= count = command_line_argument_count ()
(argument_count_from 0i32 = count, argument_count_from count) |> _assert_eq (!+, 0i32)

__assert_eq { expected = (true, 0) }

0

In [ ]:
:= fsharp_entry_name () : string = $'System.Reflection.Assembly.GetEntryAssembly().GetName().Name'

:~ get_entry_assembly_name () : string =
    run_target \|
        | Rust => \ () => (~~ "CARGO_PKG_NAME") |> get_environment_variable
        | Fsharp _ => \ () => fsharp_entry_name ()
        | target => \ () => "env.get_entry_assembly_name / target: " ++#? target

:! env_path_buf = $'std::path::PathBuf'

:= rust_current_exe () : optionm'::option' env_path_buf = $'std::env::current_exe().ok()' : optionm'::option' env_path_buf

:= rust_path_text (path : env_path_buf) : string = $'Rc::<str>::from(!path.to_string_lossy().as_ref())' : string

:~ current_executable () : string =
    run_target \|
        | Rust => \ () =>
            ?= rust_current_exe () |> optionm'::map' rust_path_text |> optionm'::unbox ?|
            | Some path => path
            | None => ""
        | Fsharp _ => \ () => $'System.Environment.ProcessPath' : string
        | target => \ () => "env.current_executable / target: " ++#? target

#### current_executable_names_an_existing_file (test)

In [10]:
///- --test --symbol
///> rust

:= path = current_executable ()
($'std::path::Path::new(&*!path).is_file()' : bool) |> _assert_eq !+

__assert_eq { expected = true }

0

In [ ]:
:~ process_id () : u64 =
    run_target \|
        | Rust => \ () => convert::u64 ($'std::process::id()' : u32)
        | Fsharp _ => \ () => convert::u64 ($'System.Environment.ProcessId' : i32)
        | _ => \ () => no_arm "env.process_id"

#### process_id_names_this_process (test)

In [11]:
///- --test --symbol
///> rust

(process_id () > 0u64) |> _assert_eq !+

__assert_eq { expected = true }

0

In [ ]:
:= append_path (path : string) : option string =
    := env_path = "PATH" |> get_environment_variable
    ? env_path = ""
    ?> None
    |:
        := env_sep = ? platform::is_windows () ?> ";" |: ":"
        path ++# env_sep ++# env_path |> Some

:~ environment_snapshot () : string =
    run_target \|
        | Rust => \ () =>
            $'Rc::<str>::from(std::env::vars_os().map(|(key, value)| format\!("{}={}", key.to_string_lossy(), \
                value.to_string_lossy())).collect::<Vec<_>>().join(&char::from(0u8).to_string()).as_str())' : string
        | target => \ () => "env.environment_snapshot / target: " ++#? target |> failwith

:= environment_snapshot_usable (snapshot : string) : bool =
    := nul = sm'::char_text 0u8
    := held = nul ++# snapshot |> sm'::to_lower
    := keeps (name : string) : bool =
        get_environment_variable name = "" || held |> sm'::contains (nul ++# sm'::to_lower name ++# "=")
    := critical =
        ? platform::is_windows ()
        ?> [ "PATH"; "SystemRoot"; "TEMP"; "TMP"; "USERPROFILE"; "LOCALAPPDATA"; "APPDATA" ]
        |: [ "PATH"; "HOME" ]
    snapshot <> "" && (critical |> listm::fold (\ kept name => kept && keeps name) !+)

:= rust_remove_var (key : string) : () = $'unsafe { std::env::remove_var(&*!key) }' : ()

:= rust_set_var (key : string) (value : string) : () = $'unsafe { std::env::set_var(&*!key, &*!value) }' : ()

:== env_entries = array_base string

:= equals_after_first (entry : string) : i32 =
    := size : i32 = sm'::length entry
    ? size < 2i32 ?> -1i32 |: sm'::index_of "=" (sm'::substring 1i32 (size - 1i32) entry)

:= entry_pair (entry : string) : string * string =
    := found = equals_after_first entry
    := size : i32 = sm'::length entry
    ? found < 0i32 ?> "", ""
    |: sm'::substring 0i32 (found + 1i32) entry, sm'::substring (found + 2i32) (size - found - 2i32) entry

:= restorable (key : string) : bool = key <> "" && not (sm'::starts_with "=" key)

:~ ^ value_in (all : env_entries) (key : string) (i : i32) : option string =
    ? i >= am'::length_base all ?> None
    |:
        := k, v = entry_pair (am'::index_base i all)
        ? k = key && restorable k ?> Some v |: value_in all key (i + 1i32)

:= extra (wanted : env_entries) (kept : string) (key : string) : bool =
    restorable key && not (sm'::starts_with kept key) && value_in wanted key 0i32 = None

:= remove_if_extra (wanted : env_entries) (kept : string) (entry : string) : () =
    := key, _ = entry_pair entry
    ? extra wanted kept key ?> rust_remove_var key

:~ ^ removed_extras (current : env_entries) (wanted : env_entries) (kept : string) (i : i32) : () =
    ? i < am'::length_base current ?>
        remove_if_extra wanted kept (am'::index_base i current)
        removed_extras current wanted kept (i + 1i32)

:= set_if_changed (current : env_entries) (entry : string) : () =
    := key, value = entry_pair entry
    ? restorable key && value_in current key 0i32 <> Some value ?> rust_set_var key value

:~ ^ set_changed (wanted : env_entries) (current : env_entries) (i : i32) : () =
    ? i < am'::length_base wanted ?>
        set_if_changed current (am'::index_base i wanted)
        set_changed wanted current (i + 1i32)

:= nul_split (text : string) = sm'::split (sm'::char_text 0u8) text

:= rust_restored (snapshot : string) (kept_prefix : string) : () =
    := wanted = nul_split snapshot
    := current = nul_split (environment_snapshot ())
    removed_extras current wanted kept_prefix 0i32
    set_changed wanted current 0i32

:~ environment_restored (snapshot : string) (kept_prefix : string) : () =
    ? environment_snapshot_usable snapshot = !-
    ?> "env.environment_restored: refusing an empty or truncated snapshot" |> failwith
    run_target_args' (snapshot, kept_prefix) \|
        | Rust => \ snapshot, kept_prefix => rust_restored snapshot kept_prefix
        | target => \ _ => "env.environment_restored / target: " ++#? target |> failwith

:= changed_after_snapshot () : () =
    rust_set_var "SPIRAL_RESTORE_CHANGED" "after"
    rust_set_var "SPIRAL_RESTORE_ADDED" "1"
    rust_set_var "SPIRAL_KEPT_NOTE" "kept"

:= snapshot_then_changed () : string =
    rust_set_var "SPIRAL_RESTORE_CHANGED" "before"
    := before = environment_snapshot ()
    changed_after_snapshot ()
    before

:= restore_test_values () : string * string * string =
    := read = get_environment_variable
    read "SPIRAL_RESTORE_CHANGED", read "SPIRAL_RESTORE_ADDED", read "SPIRAL_KEPT_NOTE"

#### restoring_a_snapshot_removes_added_and_resets_changed_variables (test)

In [12]:
///- --test --symbol
///> rust

environment_restored (snapshot_then_changed ()) "SPIRAL_KEPT_"
restore_test_values () |> _assert_eq ("before", "", "kept")

__assert_eq { expected = (before, , kept) }

0

In [ ]:
:= empty_snapshot_refused () : bool =
    catch_unwind_rust
        (\ () =>
            environment_restored "" "SPIRAL_KEPT_"
            !-
        )
        \ _ => !+

#### an_empty_snapshot_is_refused_and_keeps_the_environment (test)

In [13]:
///- --test --symbol
///> rust

(empty_snapshot_refused (), "PATH" |> get_environment_variable <> "") |> _assert_eq (!+, !+)

__assert_eq { expected = (true, true) }

0

#### a_snapshot_without_its_anchor_entry_is_unusable (test)

In [14]:
///- --test --symbol
///> rust

(environment_snapshot_usable "", environment_snapshot_usable "PATH2=x") |> _assert_eq (!-, !-)

__assert_eq { expected = (false, false) }

0

In [ ]:
:= path_first_snapshot (current : string) : string =
    "PATH=" ++# get_environment_variable "PATH" ++# sm'::char_text 0u8 ++# current

#### a_snapshot_is_usable_only_with_its_anchor_entry (test)

In [15]:
///- --test --symbol
///> rust

:= current = environment_snapshot ()
(environment_snapshot_usable (path_first_snapshot current), environment_snapshot_usable current)
|> _assert_eq (!+, !+)

__assert_eq { expected = (true, true) }

0

In [16]:
:= restored_to_itself (before : string) : bool =
    environment_restored before "SPIRAL_KEPT_"
    environment_snapshot () = before

#### an_environment_snapshot_restores_to_itself (test)

In [17]:
///- --test --symbol
///> rust

:= before = environment_snapshot ()
(sm'::length before > 0i32, restored_to_itself before, "PATH2" |> get_environment_variable)
|> _assert_eq (!+, !+, "")

__assert_eq { expected = (true, true, ) }

0